<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_9%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_GeminiAPI_LangChain_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9주차 | Gemini API & LangChain 실습

> **목표**: Gemini API(무료)로 LLM을 호출하고, LangChain으로 파이프라인을 구성한다

| 파트 | 내용 |
|------|------|
| Part 1 | Gemini API 기초 (`google-genai` SDK) |
| Part 2 | API로 텍스트 분석 (감성 분류, JSON 추출) |
| Part 3 | LangChain 기초 (LCEL: `prompt \| llm \| parser`) |
| Part 4 | LangChain 심화 (JSON, batch, 다단계) |
| Part 5 | 학생 실습 |

> ⚠️ **주의**: `google-generativeai`(구버전)는 한국어 인코딩 에러가 있습니다.  
> 반드시 `google-genai`(신버전)를 사용하세요.

---
## Part 1. Gemini API 기초

API Key는 https://aistudio.google.com 에서 무료로 발급받으세요.  
Google 계정만 있으면 됩니다. (신용카드 불필요)

In [1]:
# 1️⃣ 설치
#
# google-genai: 신규 공식 SDK (2025~)
# google-generativeai: 구버전 (deprecated, 한국어 인코딩 에러 있음!)

!pip install -q google-genai langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.9 MB/s eta 0:00:00


In [13]:
# 2️⃣ API Key 설정
#
# https://aistudio.google.com → "Get API key" → "Create API key"

import os
os.environ["GOOGLE_API_KEY"] = ""

# ⚠️ 키를 GitHub 등에 올리지 마세요!

In [5]:
# 3️⃣ Gemini API 첫 호출
from google import genai

# 클라이언트 생성 (환경변수에서 키를 자동으로 읽음)
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 텍스트 생성!
response = client.models.generate_content(
    model="gemini-2.5-flash",          # 무료 모델
    contents="텍스트 분석이 뭔지 한 줄로 설명해줘",  # 질문
)

print(f"응답: {response.text}")

응답: 텍스트 분석은 텍스트에서 유의미한 정보를 찾아내는 작업입니다.


In [6]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='텍스트 분석은 텍스트에서 유의미한 정보를 찾아내는 작업입니다.'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='6e0batuIEcu4sOIPiraMwAs' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=18,
  prompt_token_count=13,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=13
    ),
  ],
  thoughts_token_count=745,
  total_token_count=776
) automatic_function_calling_history=[] parsed=None


In [7]:
# 4️⃣ System Instruction + Temperature 설정
#
# system_instruction: 모델의 역할/규칙 (8주차 프롬프트의 "역할 부여")
# temperature: 창의성 조절 (0=보수적, 1=다양, 2=매우 창의적)

from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="리뷰: 이 영화 정말 재미있다",
    config=types.GenerateContentConfig(
        system_instruction="영화 리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요.",
        temperature=0,             # 분류는 항상 같은 답이 나오도록
        max_output_tokens=100,     # 최대 생성 길이
    ),
)

print(f"감성: {response.text}")

감성: 긍정


In [8]:
# 5️⃣ 여러 리뷰를 분류해보기
from google.genai import types

reviews = [
    "배우 연기가 훌륭하고 스토리도 좋다",
    "시간 낭비 다시는 안 봄",
    "그저 그랬다 볼만은 했음",
    "배우 연기는 좋은데 스토리가 아쉽다",
]

print("=== Gemini 감성 분류 ===")
for review in reviews:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction="리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요. 한 단어만 출력.",
            temperature=0,
        ),
    )
    print(f" {review} →  {response.text.strip()}")

=== Gemini 감성 분류 ===
 배우 연기가 훌륭하고 스토리도 좋다 →  긍정
 시간 낭비 다시는 안 봄 →  부정
 그저 그랬다 볼만은 했음 →  중립
 배우 연기는 좋은데 스토리가 아쉽다 →  THOUGHTS:
The user wants to classify the sentiment of a review as positive, negative, or neutral.
The review is: "배우 연기는 좋은데 스토리가 아쉽다" (Actor's performance is good, but the story is disappointing).

1.  **Identify positive aspects:** "배우 연기는 좋다" (Actor's performance is good) -> Positive sentiment.
2.  **Identify negative aspects:** "스토리가 아쉽다" (The story is disappointing/regrettable) -> Negative sentiment.
3.  **Evaluate the overall tone:** The conjunction "은/는...데" (but/however) often indicates a contrast where the latter part might carry more weight in determining the overall feeling, especially if it's a critical flaw. "아쉽다" (disappointing, regrettable) is a clear negative sentiment regarding a core aspect (story). While the acting is good, the story being disappointing often leads to an overall less satisfying experience.

If a review has both strong positive and strong 

---
## Part 2. API로 텍스트 분석

In [14]:
# 6️⃣ 함수로 감성 분류기 만들기
from google import genai
from google.genai import types

def classify_sentiment(review):
    """리뷰 텍스트를 받아 감성을 분류합니다"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction="리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요.",
            temperature=0,
        ),
    )
    return response.text.strip()


# 테스트
test_reviews = [
    "가격 대비 만족합니다",
    "포장이 엉망이고 제품이 파손됐어요",
    "디자인은 좋은데 내구성이 약해요",
]

for review in test_reviews:
    result = classify_sentiment(review)
    print(f"  {review}  →  {result}")

  가격 대비 만족합니다  →  긍정
  포장이 엉망이고 제품이 파손됐어요  →  부정
  디자인은 좋은데 내구성이 약해요  →  복합


In [15]:
# 7️⃣ JSON 구조화 출력 — 정보 추출
import json

def analyze_review(review):
    """리뷰에서 감성, 키워드, 요약을 JSON으로 추출합니다"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"리뷰: {review}",
        config=types.GenerateContentConfig(
            system_instruction=(
                "리뷰를 분석하여 아래 JSON 형식으로만 응답하세요. "
                "다른 텍스트 없이 순수 JSON만 출력하세요.\n"
                '{"sentiment": "긍정/부정/중립", "keywords": ["키워드1", "키워드2"], "summary": "한 줄 요약"}'
            ),
            temperature=0,
        ),
    )
    # JSON 파싱 (```json ... ``` 태그 제거)
    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)


result = analyze_review("배우 연기는 훌륭한데 스토리가 진부하고 결말이 허무했다")

print("=== 리뷰 분석 결과 (JSON) ===")
print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\n감성: {result["sentiment"]}")
print(f"키워드: {result["keywords"]}")

=== 리뷰 분석 결과 (JSON) ===
{
  "sentiment": "부정",
  "keywords": [
    "배우 연기",
    "스토리",
    "진부",
    "결말",
    "허무"
  ],
  "summary": "배우 연기는 훌륭했으나 진부한 스토리와 허무한 결말이 아쉬웠다."
}

감성: 부정
키워드: ['배우 연기', '스토리', '진부', '결말', '허무']


---
## Part 3. LangChain 기초 — LCEL 문법

LangChain = LLM 앱 개발 프레임워크  
핵심: **LCEL (LangChain Expression Language)**

```python
chain = prompt | llm | parser    # 파이프(|)로 연결!
result = chain.invoke({"변수": "값"})
```

왼쪽의 출력 → 오른쪽의 입력으로 자동 전달됩니다.

In [16]:
# 8️⃣ LangChain 기본 — prompt | llm | parser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# (1) 프롬프트 템플릿: {review}가 변수
prompt = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요."),
    ("user", "리뷰: {review}"),
])

# (2) LLM: Gemini 2.5 Flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# (3) 출력 파서: 문자열 그대로
parser = StrOutputParser()

# (4) 체인 연결! (LCEL 핵심 문법)
chain = prompt | llm | parser

# 실행
result = chain.invoke({"review": "이 영화 정말 재미있다"})
print(f"결과: {result}")

결과: 긍정


In [17]:
# 9️⃣ 각 단계가 하는 일을 확인
#
# chain = prompt | llm | parser 가
# 내부적으로 3단계를 순서대로 실행합니다.

# [Step 1] prompt: 변수를 채워서 메시지 생성
messages = prompt.invoke({"review": "이 영화 정말 재미있다"})
print("[Step 1] prompt.invoke() →")
for msg in messages.messages:
    print(f"  {msg.type}: {msg.content}")

# [Step 2] llm: 메시지를 Gemini에 보내고 응답
llm_response = llm.invoke(messages)
print(f"\n[Step 2] llm.invoke() →")
print(f"  타입: {type(llm_response).__name__}")
print(f"  내용: {llm_response.content}")

# [Step 3] parser: AIMessage에서 문자열만 추출
parsed = parser.invoke(llm_response)
print(f"\n[Step 3] parser.invoke() →")
print(f"  {parsed}")

print(f"\n→ chain.invoke()는 이 3단계를 한 번에 실행합니다!")

[Step 1] prompt.invoke() →
  system: 리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요.
  human: 리뷰: 이 영화 정말 재미있다

[Step 2] llm.invoke() →
  타입: AIMessage
  내용: 긍정

[Step 3] parser.invoke() →
  긍정

→ chain.invoke()는 이 3단계를 한 번에 실행합니다!


In [18]:
# 🔟 여러 변수 사용 + 역할 바꾸기

# system 메시지에도 변수를 쓸 수 있습니다
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 한국어로 간결하게 답변하세요."),
    ("user", "{question}"),
])

chain = prompt | llm | StrOutputParser()

# 같은 질문, 다른 역할
question = "인공지능의 미래는?"

for role in ["데이터 사이언티스트", "초등학교 선생님", "SF 소설가"]:
    result = chain.invoke({"role": role, "question": question})
    print(f"[{role}]")
    print(f"  {result[:100]}...")
    print()

[데이터 사이언티스트]
  인공지능(AI)의 미래는 매우 밝고 혁신적일 것으로 예상됩니다. 몇 가지 주요 전망은 다음과 같습니다.

*   **더욱 발전된 지능:** AI는 인간의 인지 능력을 뛰어넘는 수준...

[초등학교 선생님]
  인공지능은 우리 생활을 더 편리하고 풍요롭게 만들 거예요. 똑똑한 로봇이 집안일을 돕고, 아픈 사람을 치료하는 데 도움을 줄 수도 있어요. 공부도 더 재미있게 할 수 있도록 도와줄...

[SF 소설가]
  인공지능의 미래는 무궁무진합니다. 인간의 지능을 뛰어넘는 초지능의 등장, 인간과 AI의 융합, 그리고 AI가 만들어갈 새로운 사회와 윤리적 문제까지, 상상하는 모든 것이 현실이 될...



In [19]:
# 1️⃣1️⃣ batch — 여러 입력을 한 번에
#
# 하나씩 호출하면 느리지만, batch로 보내면 병렬 처리!
# (무료 Tier는 분당 10회 제한 → 너무 많으면 time.sleep 필요)

prompt = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감성을 긍정/부정/중립 중 하나만 답하세요."),
    ("user", "리뷰: {review}"),
])
chain = prompt | llm | StrOutputParser()

reviews = [
    {"review": "이 영화 정말 재미있다"},
    {"review": "시간 낭비 최악이다"},
    {"review": "그저 그랬다"},
    {"review": "배우 연기가 훌륭하다"},
    {"review": "지루하고 별로다"},
]

results = chain.batch(reviews)

print("=== batch 결과 ===")
for r, result in zip(reviews, results):
    print(f"  {r["review"]:20s} → {result}")

=== batch 결과 ===
  이 영화 정말 재미있다         → 긍정
  시간 낭비 최악이다           → 부정
  그저 그랬다               → 중립
  배우 연기가 훌륭하다          → 긍정
  지루하고 별로다             → 부정


---
## Part 4. LangChain 심화

In [20]:
# 1️⃣2️⃣ JSON 출력 파싱
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """리뷰를 분석하여 아래 JSON으로만 응답하세요. 다른 텍스트 없이 JSON만.
{{"sentiment": "긍정/부정/중립", "keywords": ["키워드1", "키워드2"], "summary": "한줄요약"}}"""),
    ("user", "리뷰: {review}"),
])

# JsonOutputParser: 문자열 → Python dict 자동 변환
chain = prompt | llm | JsonOutputParser()

result = chain.invoke({"review": "배우 연기는 좋은데 스토리가 너무 진부하다"})

print(f"타입: {type(result)}")   # dict!
print(f"감성:   {result["sentiment"]}")
print(f"키워드: {result["keywords"]}")
print(f"요약:   {result["summary"]}")

타입: <class 'dict'>
감성:   부정
키워드: ['배우 연기', '진부한 스토리']
요약:   배우들의 연기는 좋았지만, 스토리가 너무 진부했다.


In [28]:
# 1️⃣3️⃣ 다단계 파이프라인 — 요약 → 번역
#
# [1단계] 긴 텍스트 → 한국어 요약
# [2단계] 한국어 요약 → 영어 번역

# 1단계: 요약 체인
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", "주어진 텍스트를 2줄로 요약하세요."),
    ("user", "{text}"),
])
chain_summary = prompt_summary | llm | StrOutputParser()

# 2단계: 번역 체인
prompt_translate = ChatPromptTemplate.from_messages([
    ("system", "주어진 한국어 텍스트를 영어로 번역하세요."),
    ("user", "{text}"),
])
chain_translate = prompt_translate | llm | StrOutputParser()

# 실행
long_text = """인공지능 기술이 빠르게 발전하면서 다양한 산업에 혁신을 가져오고 있다.
특히 자연어 처리 분야에서는 대규모 언어 모델이 등장하면서
번역, 요약, 대화 시스템 등에서 놀라운 성능을 보이고 있다.
하지만 할루시네이션, 편향성 등의 문제도 해결해야 할 과제로 남아있다."""

# 1단계
summary = chain_summary.invoke({"text": long_text})
print("[1단계] 요약:")
print(f"  {summary}")

# 2단계 (1단계 결과를 입력으로!)
translation = chain_translate.invoke({"text": summary})
print(f"\n[2단계] 영어 번역:")
print(f"  {translation}")

[1단계] 요약:
  인공지능 기술은 빠르게 발전하며 다양한 산업에 혁신을 가져오고, 특히 자연어 처리 분야에서 놀라운 성능을 보인다.
하지만 할루시네이션, 편향성 등의 문제 해결이 과제로 남아있다.

[2단계] 영어 번역:
  AI technology is rapidly advancing, bringing innovation to diverse industries, and particularly demonstrating remarkable performance in the field of natural language processing. However, addressing issues such as hallucination and bias remains a challenge.


---
## Part 5. 학생 실습 ✏️

### 실습 1: Gemini API로 뉴스 카테고리 분류

In [22]:
# ✏️ 실습 1: 뉴스 기사 카테고리 분류기
#
# 요구사항:
# 1. system_instruction으로 역할 설정
# 2. 카테고리: 정치/경제/사회/문화/IT 중 하나
# 3. 아래 3개 기사를 분류하세요

articles = [
    "삼성전자가 새로운 AI 반도체 칩을 발표했다. 성능이 기존 대비 2배 향상됐다.",
    "정부가 내년도 예산안을 국회에 제출했다. 복지 분야 예산이 크게 늘었다.",
    "BTS의 새 앨범이 빌보드 차트 1위를 기록했다.",
]

# 여기에 코드를 작성하세요


In [23]:
# ✅ 실습 1 정답
from google.genai import types

for article in articles:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"기사: {article}",
        config=types.GenerateContentConfig(
            system_instruction="뉴스 기사를 정치/경제/사회/문화/IT 중 하나로 분류하세요. 카테고리만 답하세요.",
            temperature=0,
        ),
    )
    print(f"  [{response.text.strip()}] {article[:30]}...")

  [IT] 삼성전자가 새로운 AI 반도체 칩을 발표했다. 성능이 ...
  [정치] 정부가 내년도 예산안을 국회에 제출했다. 복지 분야 예...
  [문화] BTS의 새 앨범이 빌보드 차트 1위를 기록했다....


### 실습 2: LangChain으로 리뷰 분석 파이프라인

In [24]:
# ✏️ 실습 2: LangChain LCEL로 리뷰 분석 파이프라인
#
# 요구사항:
# 1. ChatPromptTemplate + ChatGoogleGenerativeAI + JsonOutputParser
# 2. 출력: {"sentiment": "...", "score": 1~5, "keywords": [...]}
# 3. batch로 아래 5개 리뷰를 한 번에 분석

test_reviews = [
    {"review": "배송 빠르고 품질 좋아요 재구매 의사 있습니다"},
    {"review": "불량품이 왔는데 환불이 안되네요 최악"},
    {"review": "가격 대비 그럭저럭 괜찮습니다"},
    {"review": "디자인이 예쁘고 사용감도 좋습니다 만족"},
    {"review": "사이즈가 안 맞아서 교환했는데 한참 걸렸어요"},
]

# 여기에 코드를 작성하세요


In [25]:
# ✅ 실습 2 정답
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """리뷰를 분석하여 아래 JSON으로만 응답하세요.
{{"sentiment": "긍정/부정/중립", "score": 1~5, "keywords": ["키워드1", "키워드2"]}}"""),
    ("user", "리뷰: {review}"),
])

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
chain = prompt | llm | JsonOutputParser()

results = chain.batch(test_reviews)

print("=== 리뷰 분석 결과 ===")
for review, result in zip(test_reviews, results):
    print(f"\n  리뷰: {review["review"]}")
    print(f"  분석: {result}")

=== 리뷰 분석 결과 ===

  리뷰: 배송 빠르고 품질 좋아요 재구매 의사 있습니다
  분석: {'sentiment': '긍정', 'score': 5, 'keywords': ['배송', '품질', '재구매']}

  리뷰: 불량품이 왔는데 환불이 안되네요 최악
  분석: {'sentiment': '부정', 'score': 1, 'keywords': ['불량품', '환불']}

  리뷰: 가격 대비 그럭저럭 괜찮습니다
  분석: {'sentiment': '중립', 'score': 3, 'keywords': ['가격', '괜찮다']}

  리뷰: 디자인이 예쁘고 사용감도 좋습니다 만족
  분석: {'sentiment': '긍정', 'score': 5, 'keywords': ['디자인', '사용감']}

  리뷰: 사이즈가 안 맞아서 교환했는데 한참 걸렸어요
  분석: {'sentiment': '부정', 'score': 2, 'keywords': ['사이즈', '교환', '교환 지연']}


### 실습 3: 다단계 파이프라인

In [26]:
# ✏️ 실습 3: 다단계 파이프라인
#
# [1단계] 리뷰 → 키워드 3개 추출
# [2단계] 키워드 → 개선 제안 1줄

review = "배송은 빠른데 포장이 엉망이고 제품에 스크래치가 있었다"

# 여기에 코드를 작성하세요


In [27]:
# ✅ 실습 3 정답
from langchain_core.output_parsers import StrOutputParser

# 1단계: 키워드 추출
prompt_kw = ChatPromptTemplate.from_messages([
    ("system", "리뷰에서 핵심 키워드 3개를 추출하세요. 쉼표로 구분, 키워드만 답하세요."),
    ("user", "리뷰: {review}"),
])
chain_kw = prompt_kw | llm | StrOutputParser()

# 2단계: 개선 제안
prompt_suggest = ChatPromptTemplate.from_messages([
    ("system", "아래 키워드를 바탕으로 서비스 개선 제안을 한 줄로 작성하세요."),
    ("user", "키워드: {keywords}"),
])
chain_suggest = prompt_suggest | llm | StrOutputParser()

# 실행
keywords = chain_kw.invoke({"review": review})
print(f"[1단계] 키워드: {keywords}")

suggestion = chain_suggest.invoke({"keywords": keywords})
print(f"[2단계] 개선 제안: {suggestion}")

[1단계] 키워드: 배송, 포장, 스크래치


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 34.761124379s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

---
## 핵심 정리

### Gemini API (`google-genai`)

```python
from google import genai
from google.genai import types

client = genai.Client(api_key="KEY")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="질문",
    config=types.GenerateContentConfig(
        system_instruction="역할",
        temperature=0,
    ),
)
print(response.text)
```

### LangChain LCEL

```python
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([...])
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"변수": "값"})      # 단건
results = chain.batch([{...}, {...}])       # 복수건
```

### 언제 무엇을 쓸까?

| 상황 | 추천 |
|------|------|
| 단순 API 호출 | `google-genai` 직접 |
| 프롬프트 재사용 | LangChain PromptTemplate |
| JSON 파싱 | LangChain JsonOutputParser |
| 여러 건 일괄 | LangChain batch |
| 다단계 | LangChain Chain 연결 |
| 나중에 모델 교체 | LangChain (llm만 바꾸면 됨) |